In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH OPTIMIZATIONS
# ============================================================================
def train_model(train_loader, val_loader, device, config):
    """
    Train model with various optimizations
    
    config: dict with hyperparameters
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights (if imbalanced)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    # Alternative: CosineAnnealingLR
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer, T_max=config['num_epochs'], eta_min=1e-6
    # )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. HYPERPARAMETER SEARCH
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Optimized grid search focused on the best performing region
    """
    param_grid = [
        # ===== BASELINE: Current Best =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== DROPOUT FINE-TUNING (Around 0.55) =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.525,  # Slightly lower
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.575,  # Slightly higher
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.60,  # Test upper bound
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== ARCHITECTURE VARIATIONS (Keeping dropout=0.55) =====
        # Deeper with same pattern
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Even deeper
        {
            'hidden_dims': [768, 384, 192, 96, 48],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Slightly wider first layer
        {
            'hidden_dims': [832, 416, 208],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # More gradual reduction
        {
            'hidden_dims': [768, 512, 256, 128],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== LEARNING RATE FINE-TUNING =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.0008,  # Slightly lower
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.0012,  # Slightly higher
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== WEIGHT DECAY VARIATIONS =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 2e-5,  # Double current
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 5e-6,  # Half current
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== BATCH SIZE VARIATIONS =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 64,  # Larger batch
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 16,  # Smaller batch (more gradient updates)
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== COMBINED OPTIMIZATIONS =====
        # Higher dropout + lower LR
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.60,
            'learning_rate': 0.0008,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Deeper + adjusted dropout
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.525,  # Slightly lower for deeper network
            'learning_rate': 0.001,
            'weight_decay': 2e-5,  # More regularization
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Wider + higher dropout
        {
            'hidden_dims': [832, 416, 208],
            'dropout_rate': 0.60,
            'learning_rate': 0.0012,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== ALTERNATIVE PATTERNS =====
        # Bottleneck architecture
        {
            'hidden_dims': [768, 256, 768, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Slower reduction
        {
            'hidden_dims': [768, 640, 512, 384, 192],
            'dropout_rate': 0.50,  # Lower for very deep network
            'learning_rate': 0.0008,
            'weight_decay': 2e-5,
            'batch_size': 32,
            'num_epochs': 150,  # More epochs for deeper network
            'early_stopping_patience': 25
        },
        # Conservative approach (less aggressive reduction)
        {
            'hidden_dims': [768, 576, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
    ]
    
    best_config = None
    best_score = 0
    results_log = []
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        
        # Log results
        results_log.append({
            'config_idx': i + 1,
            'f1_score': f1,
            'config': config
        })
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    # Print summary of all results
    print(f"\n{'='*60}")
    print("SUMMARY OF ALL CONFIGURATIONS:")
    print(f"{'='*60}")
    results_log.sort(key=lambda x: x['f1_score'], reverse=True)
    for i, result in enumerate(results_log[:5]):  # Top 5
        print(f"\n{i+1}. F1: {result['f1_score']:.4f}")
        print(f"   Config {result['config_idx']}: {result['config']}")
    
    print(f"\n{'='*60}")
    print("BEST CONFIGURATION:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config
# ============================================================================
# 5. CROSS-VALIDATION FOR ROBUST EVALUATION
# ============================================================================
def cross_validate_model(data, device, config, n_folds=5):
    """
    K-fold cross-validation for more robust performance estimate
    """
    labels = [d['label'] for d in data]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        train_data = [data[i] for i in train_idx]
        val_data = [data[i] for i in val_idx]
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        fold_scores.append(f1)
    
    print(f"\n{'='*60}")
    print("Cross-Validation Results:")
    print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*60}")
    
    return fold_scores

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    
    # Option 2: Hyperparameter search (uncomment to use)
    best_config = hyperparameter_search(data, device)
    
    # Option 3: Cross-validation (uncomment to use)
    #cv_scores = cross_validate_model(data, device, best_config, n_folds=5)
    
    # Final training on full data with train/val split
    print("\nTraining final model...")
    train_data, val_data = train_test_split(
        data, test_size=0.2, random_state=42,
        stratify=[d['label'] for d in data]
    )
    
    train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
    val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=False
    )
    
    model, best_f1, train_losses, val_f1s = train_model(
        train_loader, val_loader, device, best_config
    )
    
    # Load best model for prediction
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Make predictions on test set
    print("\nMaking predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    test_dataset = EmbeddingDataset(
        test_data, 
        scaler=train_dataset.scaler, 
        is_test=True
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    predictions = []
    test_ids = []
    
    with torch.no_grad():
        for features, ids in test_loader:
            features = features.to(device)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(preds)
            # Convert tensor IDs to regular integers/numbers
            if isinstance(ids, torch.Tensor):
                test_ids.extend(ids.cpu().numpy().tolist())
            else:
                test_ids.extend(ids)
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': predictions
    })
    submission.to_csv('ann_v7.csv', index=False)
    print("\n✓ Submission file created: submission.csv")
    print(f"Predictions: 0={predictions.count(0)}, 1={predictions.count(1)}")

if __name__ == "__main__":
    main()

Using device: cpu

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204

Testing configuration 1/20
{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.55, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 120, 'early_stopping_patience': 20}
Epoch 1/120
  Train Loss: 2.8134, Train F1: 0.4978
  Val Loss: 0.6177, Val F1: 0.6694
  LR: 0.001000
  ✓ New best F1: 0.6694
  ✓ New best F1: 0.6695
  ✓ New best F1: 0.6826
Epoch 5/120
  Train Loss: 1.3346, Train F1: 0.6752
  Val Loss: 1.0485, Val F1: 0.6627
  LR: 0.001000
  ✓ New best F1: 0.6858
  ✓ New best F1: 0.7058
Epoch 10/120
  Train Loss: 0.8599, Train F1: 0.7432
  Val Loss: 1.0908, Val F1: 0.5903
  LR: 0.001000
Epoch 15/120
  Train Loss: 0.4355, Train F1: 0.8301
  Val Loss: 1.0183, Val F1: 0.6325
  LR: 0.000500
Epoch 20/120
  Train Loss: 0.2630, Train F1: 0.8662
  Val Loss: 1.1745, Val F1: 0.6096
  LR: 0.000250
  → Learning rate reduced from 0.000500 to 0.000250
Epoch 25/120
  Train Los

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Epoch 1/120
  Train Loss: 8.6544, Train F1: 0.3825
  Val Loss: 1.6826, Val F1: 0.5948
  LR: 0.001000
  ✓ New best F1: 0.5948
  ✓ New best F1: 0.6941
  ✓ New best F1: 0.7434
Epoch 5/120
  Train Loss: 1.6982, Train F1: 0.6280
  Val Loss: 1.0240, Val F1: 0.7009
  LR: 0.001000
Epoch 10/120
  Train Loss: 0.9861, Train F1: 0.7551
  Val Loss: 1.1793, Val F1: 0.6774
  LR: 0.000500
  → Learning rate reduced from 0.001000 to 0.000500
Epoch 15/120
  Train Loss: 0.5291, Train F1: 0.8224
  Val Loss: 1.5006, Val F1: 0.6336
  LR: 0.000500
Epoch 20/120
  Train Loss: 0.3383, Train F1: 0.8784
  Val Loss: 1.4121, Val F1: 0.6510
  LR: 0.000250

Early stopping triggered after 24 epochs

Training completed. Best Val F1: 0.7434

Final Validation Classification Report:
               precision    recall  f1-score   support

Not Important       0.89      0.96      0.93       265
    Important       0.52      0.27      0.35        41

     accuracy                           0.87       306
    macro avg       0.

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Epoch 1/120
  Train Loss: 5.1028, Train F1: 0.4508
  Val Loss: 0.5979, Val F1: 0.7184
  LR: 0.000800
  ✓ New best F1: 0.7184
Epoch 5/120
  Train Loss: 1.4474, Train F1: 0.6186
  Val Loss: 0.8219, Val F1: 0.6488
  LR: 0.000800
Epoch 10/120
  Train Loss: 0.9053, Train F1: 0.7113
  Val Loss: 0.9448, Val F1: 0.6845
  LR: 0.000400
Epoch 15/120
  Train Loss: 0.8608, Train F1: 0.7782
  Val Loss: 1.0815, Val F1: 0.6264
  LR: 0.000200
Epoch 20/120
  Train Loss: 0.6096, Train F1: 0.8115
  Val Loss: 1.1319, Val F1: 0.6434
  LR: 0.000100

Early stopping triggered after 21 epochs

Training completed. Best Val F1: 0.7184

Final Validation Classification Report:
               precision    recall  f1-score   support

Not Important       0.90      0.96      0.93       265
    Important       0.52      0.29      0.38        41

     accuracy                           0.87       306
    macro avg       0.71      0.63      0.65       306
 weighted avg       0.85      0.87      0.85       306


Testing co

In [2]:
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import pandas as pd

# ============================================================================
# 1. OPTIMIZED DATASET WITH NORMALIZATION
# ============================================================================
class EmbeddingDataset(Dataset):
    def __init__(self, data, scaler=None, is_test=False, fit_scaler=False):
        self.data = data
        self.is_test = is_test
        
        # Extract features
        features = []
        for record in data:
            feat = record['image_embedding'] + record['text_embedding']
            features.append(feat)
        
        features = np.array(features)
        
        # Normalize features
        if fit_scaler:
            self.scaler = StandardScaler()
            features = self.scaler.fit_transform(features)
        elif scaler is not None:
            self.scaler = scaler
            features = self.scaler.transform(features)
        else:
            self.scaler = None
        
        self.features = features
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.FloatTensor(self.features[idx])
        
        if self.is_test:
            return features, self.data[idx]['id']
        else:
            label = torch.LongTensor([self.data[idx]['label']])[0]
            return features, label

# ============================================================================
# 2. IMPROVED MODEL ARCHITECTURE
# ============================================================================
class OptimizedEarlyFusionMLP(nn.Module):
    def __init__(self, input_dim=1024, hidden_dims=[512, 256, 128, 64], 
                 dropout_rate=0.4):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate if i < len(hidden_dims) - 1 else dropout_rate * 0.5)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, 2))
        self.model = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        return self.model(x)

# ============================================================================
# 3. TRAINING FUNCTION WITH OPTIMIZATIONS
# ============================================================================
def train_model(train_loader, val_loader, device, config):
    """
    Train model with various optimizations
    
    config: dict with hyperparameters
    """
    model = OptimizedEarlyFusionMLP(
        hidden_dims=config['hidden_dims'],
        dropout_rate=config['dropout_rate']
    ).to(device)
    
    # Loss function with class weights (if imbalanced)
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer with weight decay (L2 regularization)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=config['learning_rate'],
        weight_decay=config['weight_decay']
    )
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='max',  # maximize F1
        factor=0.5, 
        patience=5,
        min_lr=1e-6
    )
    
    # Alternative: CosineAnnealingLR
    # scheduler = optim.lr_scheduler.CosineAnnealingLR(
    #     optimizer, T_max=config['num_epochs'], eta_min=1e-6
    # )
    
    best_f1 = 0
    patience_counter = 0
    train_losses = []
    val_f1_scores = []
    
    for epoch in range(config['num_epochs']):
        # Training phase
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []
        
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_labels.extend(labels.cpu().numpy())
        
        avg_train_loss = train_loss / len(train_loader)
        train_f1 = f1_score(train_labels, train_preds, average='macro')
        
        # Validation phase
        model.eval()
        val_preds = []
        val_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                preds = torch.argmax(outputs, dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_labels.extend(labels.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        
        train_losses.append(avg_train_loss)
        val_f1_scores.append(val_f1)
        
        # Get current learning rate before scheduler step
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update learning rate
        old_lr = current_lr
        scheduler.step(val_f1)
        new_lr = optimizer.param_groups[0]['lr']
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}/{config['num_epochs']}")
            print(f"  Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}")
            print(f"  Val Loss: {avg_val_loss:.4f}, Val F1: {val_f1:.4f}")
            print(f"  LR: {new_lr:.6f}")
            if new_lr < old_lr:
                print(f"  → Learning rate reduced from {old_lr:.6f} to {new_lr:.6f}")
        
        # Save best model
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_f1': best_f1,
                'config': config
            }, 'best_model.pth')
            patience_counter = 0
            print(f"  ✓ New best F1: {best_f1:.4f}")
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= config['early_stopping_patience']:
            print(f"\nEarly stopping triggered after {epoch+1} epochs")
            break
    
    print(f"\nTraining completed. Best Val F1: {best_f1:.4f}")
    
    # Print final classification report
    print("\nFinal Validation Classification Report:")
    print(classification_report(val_labels, val_preds, 
                                target_names=['Not Important', 'Important']))
    
    return model, best_f1, train_losses, val_f1_scores

# ============================================================================
# 4. HYPERPARAMETER SEARCH
# ============================================================================
def hyperparameter_search(train_data, device):
    """
    Optimized grid search focused on the best performing region
    """
    param_grid = [
        # ===== BASELINE: Current Best =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== DROPOUT FINE-TUNING (Around 0.55) =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.525,  # Slightly lower
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.575,  # Slightly higher
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.60,  # Test upper bound
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== ARCHITECTURE VARIATIONS (Keeping dropout=0.55) =====
        # Deeper with same pattern
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Even deeper
        {
            'hidden_dims': [768, 384, 192, 96, 48],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Slightly wider first layer
        {
            'hidden_dims': [832, 416, 208],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # More gradual reduction
        {
            'hidden_dims': [768, 512, 256, 128],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== LEARNING RATE FINE-TUNING =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.0008,  # Slightly lower
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.0012,  # Slightly higher
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== WEIGHT DECAY VARIATIONS =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 2e-5,  # Double current
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 5e-6,  # Half current
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== BATCH SIZE VARIATIONS =====
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 64,  # Larger batch
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 16,  # Smaller batch (more gradient updates)
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== COMBINED OPTIMIZATIONS =====
        # Higher dropout + lower LR
        {
            'hidden_dims': [768, 384, 192],
            'dropout_rate': 0.60,
            'learning_rate': 0.0008,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Deeper + adjusted dropout
        {
            'hidden_dims': [768, 384, 192, 96],
            'dropout_rate': 0.525,  # Slightly lower for deeper network
            'learning_rate': 0.001,
            'weight_decay': 2e-5,  # More regularization
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Wider + higher dropout
        {
            'hidden_dims': [832, 416, 208],
            'dropout_rate': 0.60,
            'learning_rate': 0.0012,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        
        # ===== ALTERNATIVE PATTERNS =====
        # Bottleneck architecture
        {
            'hidden_dims': [768, 256, 768, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
        # Slower reduction
        {
            'hidden_dims': [768, 640, 512, 384, 192],
            'dropout_rate': 0.50,  # Lower for very deep network
            'learning_rate': 0.0008,
            'weight_decay': 2e-5,
            'batch_size': 32,
            'num_epochs': 150,  # More epochs for deeper network
            'early_stopping_patience': 25
        },
        # Conservative approach (less aggressive reduction)
        {
            'hidden_dims': [768, 576, 384, 192],
            'dropout_rate': 0.55,
            'learning_rate': 0.001,
            'weight_decay': 1e-5,
            'batch_size': 32,
            'num_epochs': 120,
            'early_stopping_patience': 20
        },
    ]
    
    best_config = None
    best_score = 0
    results_log = []
    
    for i, config in enumerate(param_grid):
        print(f"\n{'='*60}")
        print(f"Testing configuration {i+1}/{len(param_grid)}")
        print(f"{'='*60}")
        print(config)
        
        # Split data
        train_split, val_split = train_test_split(
            train_data, test_size=0.2, random_state=42, 
            stratify=[d['label'] for d in train_data]
        )
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_split, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_split, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        
        # Log results
        results_log.append({
            'config_idx': i + 1,
            'f1_score': f1,
            'config': config
        })
        
        if f1 > best_score:
            best_score = f1
            best_config = config
            print(f"\n✓ New best configuration! F1: {f1:.4f}")
    
    # Print summary of all results
    print(f"\n{'='*60}")
    print("SUMMARY OF ALL CONFIGURATIONS:")
    print(f"{'='*60}")
    results_log.sort(key=lambda x: x['f1_score'], reverse=True)
    for i, result in enumerate(results_log[:5]):  # Top 5
        print(f"\n{i+1}. F1: {result['f1_score']:.4f}")
        print(f"   Config {result['config_idx']}: {result['config']}")
    
    print(f"\n{'='*60}")
    print("BEST CONFIGURATION:")
    print(best_config)
    print(f"Best F1 Score: {best_score:.4f}")
    print(f"{'='*60}")
    
    return best_config
# ============================================================================
# 5. CROSS-VALIDATION FOR ROBUST EVALUATION
# ============================================================================
def cross_validate_model(data, device, config, n_folds=5):
    """
    K-fold cross-validation for more robust performance estimate
    """
    labels = [d['label'] for d in data]
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(data, labels)):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print(f"{'='*60}")
        
        train_data = [data[i] for i in train_idx]
        val_data = [data[i] for i in val_idx]
        
        # Create datasets
        train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
        val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['batch_size'], 
            shuffle=True
        )
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['batch_size'], 
            shuffle=False
        )
        
        # Train model
        _, f1, _, _ = train_model(train_loader, val_loader, device, config)
        fold_scores.append(f1)
    
    print(f"\n{'='*60}")
    print("Cross-Validation Results:")
    print(f"Fold Scores: {[f'{s:.4f}' for s in fold_scores]}")
    print(f"Mean F1: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    print(f"{'='*60}")
    
    return fold_scores

# ============================================================================
# 6. MAIN TRAINING PIPELINE
# ============================================================================
def main():
    # Set random seeds for reproducibility
    torch.manual_seed(42)
    np.random.seed(42)
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load data
    print("\nLoading training data...")
    with open('train_part1.json', 'r') as f:
        data = json.load(f)
    
    print(f"Total samples: {len(data)}")
    labels = [d['label'] for d in data]
    print(f"Class distribution: 0={labels.count(0)}, 1={labels.count(1)}")
    
    # Option 2: Hyperparameter search (uncomment to use)
    best_config = hyperparameter_search(data, device)
    
    # Option 3: Cross-validation (uncomment to use)
    cv_scores = cross_validate_model(data, device, best_config, n_folds=5)
    
    # Final training on full data with train/val split
    print("\nTraining final model...")
    train_data, val_data = train_test_split(
        data, test_size=0.2, random_state=42,
        stratify=[d['label'] for d in data]
    )
    
    train_dataset = EmbeddingDataset(train_data, fit_scaler=True)
    val_dataset = EmbeddingDataset(val_data, scaler=train_dataset.scaler)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=best_config['batch_size'], 
        shuffle=False
    )
    
    model, best_f1, train_losses, val_f1s = train_model(
        train_loader, val_loader, device, best_config
    )
    
    # Load best model for prediction
    checkpoint = torch.load('best_model.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    
    # Make predictions on test set
    print("\nMaking predictions on test set...")
    with open('test.json', 'r') as f:
        test_data = json.load(f)
    
    test_dataset = EmbeddingDataset(
        test_data, 
        scaler=train_dataset.scaler, 
        is_test=True
    )
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
    
    predictions = []
    test_ids = []
    
    with torch.no_grad():
        for features, ids in test_loader:
            features = features.to(device)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            predictions.extend(preds)
            # Convert tensor IDs to regular integers/numbers
            if isinstance(ids, torch.Tensor):
                test_ids.extend(ids.cpu().numpy().tolist())
            else:
                test_ids.extend(ids)
    
    # Create submission
    submission = pd.DataFrame({
        'row_id': test_ids,
        'target': predictions
    })
    submission.to_csv('ann_v8.csv', index=False)
    print("\n✓ Submission file created: submission.csv")
    print(f"Predictions: 0={predictions.count(0)}, 1={predictions.count(1)}")

if __name__ == "__main__":
    main()
    

Using device: cpu

Loading training data...
Total samples: 1530
Class distribution: 0=1326, 1=204

Testing configuration 1/20
{'hidden_dims': [768, 384, 192], 'dropout_rate': 0.55, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'batch_size': 32, 'num_epochs': 120, 'early_stopping_patience': 20}
Epoch 1/120
  Train Loss: 2.8134, Train F1: 0.4978
  Val Loss: 0.6177, Val F1: 0.6694
  LR: 0.001000
  ✓ New best F1: 0.6694
  ✓ New best F1: 0.6695
  ✓ New best F1: 0.6826
Epoch 5/120
  Train Loss: 1.3346, Train F1: 0.6752
  Val Loss: 1.0485, Val F1: 0.6627
  LR: 0.001000
  ✓ New best F1: 0.6858
  ✓ New best F1: 0.7058
Epoch 10/120
  Train Loss: 0.8599, Train F1: 0.7432
  Val Loss: 1.0908, Val F1: 0.5903
  LR: 0.001000
Epoch 15/120
  Train Loss: 0.4355, Train F1: 0.8301
  Val Loss: 1.0183, Val F1: 0.6325
  LR: 0.000500
Epoch 20/120
  Train Loss: 0.2630, Train F1: 0.8662
  Val Loss: 1.1745, Val F1: 0.6096
  LR: 0.000250
  → Learning rate reduced from 0.000500 to 0.000250
Epoch 25/120
  Train Los

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Epoch 1/120
  Train Loss: 8.6544, Train F1: 0.3825
  Val Loss: 1.6826, Val F1: 0.5948
  LR: 0.001000
  ✓ New best F1: 0.5948
  ✓ New best F1: 0.6941
  ✓ New best F1: 0.7434
Epoch 5/120
  Train Loss: 1.6982, Train F1: 0.6280
  Val Loss: 1.0240, Val F1: 0.7009
  LR: 0.001000
Epoch 10/120
  Train Loss: 0.9861, Train F1: 0.7551
  Val Loss: 1.1793, Val F1: 0.6774
  LR: 0.000500
  → Learning rate reduced from 0.001000 to 0.000500
Epoch 15/120
  Train Loss: 0.5291, Train F1: 0.8224
  Val Loss: 1.5006, Val F1: 0.6336
  LR: 0.000500
Epoch 20/120
  Train Loss: 0.3383, Train F1: 0.8784
  Val Loss: 1.4121, Val F1: 0.6510
  LR: 0.000250

Early stopping triggered after 24 epochs

Training completed. Best Val F1: 0.7434

Final Validation Classification Report:
               precision    recall  f1-score   support

Not Important       0.89      0.96      0.93       265
    Important       0.52      0.27      0.35        41

     accuracy                           0.87       306
    macro avg       0.

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Epoch 1/120
  Train Loss: 5.1028, Train F1: 0.4508
  Val Loss: 0.5979, Val F1: 0.7184
  LR: 0.000800
  ✓ New best F1: 0.7184
Epoch 5/120
  Train Loss: 1.4474, Train F1: 0.6186
  Val Loss: 0.8219, Val F1: 0.6488
  LR: 0.000800
Epoch 10/120
  Train Loss: 0.9053, Train F1: 0.7113
  Val Loss: 0.9448, Val F1: 0.6845
  LR: 0.000400
Epoch 15/120
  Train Loss: 0.8608, Train F1: 0.7782
  Val Loss: 1.0815, Val F1: 0.6264
  LR: 0.000200
Epoch 20/120
  Train Loss: 0.6096, Train F1: 0.8115
  Val Loss: 1.1319, Val F1: 0.6434
  LR: 0.000100

Early stopping triggered after 21 epochs

Training completed. Best Val F1: 0.7184

Final Validation Classification Report:
               precision    recall  f1-score   support

Not Important       0.90      0.96      0.93       265
    Important       0.52      0.29      0.38        41

     accuracy                           0.87       306
    macro avg       0.71      0.63      0.65       306
 weighted avg       0.85      0.87      0.85       306


Testing co